# Day 2 — Categorical Encoding
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

ML models only understand numbers. This notebook covers:
- Ordinal encoding for ordered categories
- One-hot encoding for nominal, low-cardinality features
- Target encoding with K-fold cross-validation (anti-leakage)
- Frequency / count encoding
- Binary encoding
- Handling unknown categories at inference time
- Encoding inside a Pipeline to prevent leakage

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

np.random.seed(42)
print('Libraries loaded.')

## 1. Create a Mixed Dataset

In [ ]:
n = 2000
np.random.seed(42)

cities = np.random.choice(
    ['London', 'New York', 'Tokyo', 'Berlin', 'Sydney', 'Mumbai',
     'Paris', 'Toronto', 'Seoul', 'Amsterdam'],
    size=n, p=[0.25, 0.20, 0.15, 0.10, 0.08, 0.07, 0.06, 0.04, 0.03, 0.02]
)

df = pd.DataFrame({
    'size':        np.random.choice(['small', 'medium', 'large', 'xlarge'], n),
    'colour':      np.random.choice(['red', 'blue', 'green'], n),
    'city':        cities,
    'age':         np.random.randint(18, 70, n).astype(float),
    'income':      np.random.exponential(50000, n),
})

# Target: binary — income > 60k
df['target'] = (df['income'] > 60000).astype(int)

print(df.head())
print('\nCategory counts:')
print(df['city'].value_counts())

## 2. Ordinal Encoding — When Order Exists

In [ ]:
# Size has a natural order: small < medium < large < xlarge
oe = OrdinalEncoder(categories=[['small', 'medium', 'large', 'xlarge']])
df['size_enc'] = oe.fit_transform(df[['size']])

print('Ordinal mapping:')
for cat, code in zip(oe.categories_[0], range(len(oe.categories_[0]))):
    print(f'  {cat} → {code}')

# Verify
print('\nSample:')
print(df[['size', 'size_enc']].drop_duplicates().sort_values('size_enc'))

## 3. One-Hot Encoding — Nominal, Low Cardinality

In [ ]:
# Colour has 3 values — OHE creates 2 columns (drop='first' to avoid dummy trap)
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
colour_encoded = ohe.fit_transform(df[['colour']])
colour_cols = ohe.get_feature_names_out(['colour'])

print('OHE feature names:', colour_cols)
print(f'Original: 3 categories → OHE: {colour_encoded.shape[1]} columns (dropped one)')

# Demo: what drop='first' drops
ohe_full = OneHotEncoder(drop=None, sparse_output=False)
full_cols = ohe_full.fit_transform(df[['colour']])
print(f'\nWithout drop=first: {full_cols.shape[1]} columns (multicollinear!)')

# Demo of handle_unknown='ignore' at inference
new_data = pd.DataFrame({'colour': ['purple']})  # unseen category
encoded_new = ohe.transform(new_data)
print(f'\nUnseen category "purple" → {encoded_new[0]} (all zeros, no crash)')

## 4. Frequency Encoding — High Cardinality, No Leakage

In [ ]:
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build frequency map from training data ONLY
freq_map = X_train['city'].value_counts().to_dict()

X_train = X_train.copy()
X_test  = X_test.copy()
X_train['city_freq'] = X_train['city'].map(freq_map)
X_test['city_freq']  = X_test['city'].map(freq_map).fillna(0)  # unseen → 0

print('Frequency encoding for city:')
print(X_train[['city', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False))

## 5. Target Encoding with K-Fold Cross-Validation

Most important anti-leakage technique for high-cardinality features.

In [ ]:
from sklearn.model_selection import KFold

def cross_fold_target_encode(X_tr, y_tr, X_te, col, n_splits=5, smoothing=1.0):
    """K-fold cross target encoding to prevent leakage."""
    global_mean = y_tr.mean()
    X_tr_enc = X_tr.copy()
    X_te_enc  = X_te.copy()

    # Train encoding: encode each row using out-of-fold statistics
    encoded_col = np.zeros(len(X_tr))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold_train_idx, fold_val_idx in kf.split(X_tr):
        X_fold = X_tr.iloc[fold_train_idx]
        y_fold = y_tr.iloc[fold_train_idx]

        # Compute target mean per category using only fold training data
        cat_means = y_fold.groupby(X_fold[col]).mean()

        # Apply smoothed encoding to validation fold
        for idx in fold_val_idx:
            cat = X_tr.iloc[idx][col]
            cat_mean = cat_means.get(cat, global_mean)
            n = (X_fold[col] == cat).sum()
            # Bayesian smoothing
            encoded_col[idx] = (n * cat_mean + smoothing * global_mean) / (n + smoothing)

    X_tr_enc[f'{col}_te'] = encoded_col

    # Test encoding: use full training statistics
    full_cat_means = y_tr.groupby(X_tr[col]).mean()
    X_te_enc[f'{col}_te'] = X_te[col].map(full_cat_means).fillna(global_mean)

    return X_tr_enc, X_te_enc

# Reset to clean data
X_train2 = X_train.copy()
X_test2  = X_test.copy()
X_train2, X_test2 = cross_fold_target_encode(X_train2, y_train, X_test2, 'city')

print('Target encoding (cross-fold) for city:')
print(X_train2[['city', 'city_te']].drop_duplicates().sort_values('city_te', ascending=False))

## 6. Comparing Encodings — Model Performance

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

results = {}

# Strategy 1: OHE city (all 10 categories)
ct_ohe = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
     ['colour', 'city', 'size']),
], remainder='passthrough')
pipe_ohe = Pipeline([('ct', ct_ohe), ('model', RandomForestClassifier(n_estimators=50, random_state=42))])
pipe_ohe.fit(X[['colour', 'city', 'size', 'age', 'income']], y)
# dummy check
cv_ohe = cross_val_score(pipe_ohe, X[['colour', 'city', 'size', 'age', 'income']], y,
                          cv=5, scoring='roc_auc').mean()
results['OHE (city)'] = cv_ohe

# Strategy 2: Frequency encoding city
from sklearn.preprocessing import FunctionTransformer

# Quick manual test of freq vs OHE performance
X_freq = X.copy()
X_freq['city_freq'] = X['city'].map(freq_map).fillna(0)
X_freq['size_ord']  = OrdinalEncoder(categories=[['small','medium','large','xlarge']]).fit_transform(X[['size']])
X_freq_ohe = pd.get_dummies(X_freq[['colour']], drop_first=True)
X_final = pd.concat([X_freq[['age', 'income', 'city_freq', 'size_ord']], X_freq_ohe], axis=1)

rf = RandomForestClassifier(n_estimators=50, random_state=42)
cv_freq = cross_val_score(rf, X_final, y, cv=5, scoring='roc_auc').mean()
results['Freq + Ordinal + OHE(colour)'] = cv_freq

print('=== Encoding Strategy Comparison (5-Fold AUC) ===')
for strat, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {strat:<40} AUC = {auc:.4f}')

## 7. Encoding Unknown Categories at Inference

In [ ]:
# Demonstrate graceful handling of unseen categories
ohe_safe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe_safe.fit(X_train[['colour']])

# Known category
known = ohe_safe.transform(pd.DataFrame({'colour': ['red']}))
print(f'Known category "red": {known[0]}')

# Unknown category — all zeros (model sees baseline)
unknown = ohe_safe.transform(pd.DataFrame({'colour': ['purple']}))
print(f'Unknown category "purple": {unknown[0]} (all zeros → treated as baseline/dropped category)')

# OrdinalEncoder also needs handle_unknown
oe_safe = OrdinalEncoder(
    categories=[['small', 'medium', 'large', 'xlarge']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
oe_safe.fit(X_train[['size']])
print(f'Unknown ordinal → -1: {oe_safe.transform(pd.DataFrame({"size": ["micro"]}))}')

## Exercises

1. **High cardinality experiment**: Generate a dataset with a 500-category feature. Compare OHE vs frequency encoding on a classification task. Measure both AUC and training time.

2. **Target encoding leakage demo**: Implement target encoding WITHOUT K-fold (fit on full training set). Compare CV AUC to the K-fold version. Quantify the leakage-induced optimism.

3. **Ordinal vs OHE**: Take a dataset with a 5-level ordinal feature. Fit a linear regression with (a) ordinal encoding and (b) OHE. Compare coefficients and prediction quality. When does ordinal encoding outperform OHE for linear models?

4. **sklearn TargetEncoder**: Use `sklearn.preprocessing.TargetEncoder` (sklearn ≥ 1.3) to encode the `city` column. Compare its AUC to your custom K-fold implementation above.

In [ ]:
# ── Exercise 1 Solution: High cardinality comparison ──
# Generate 500-category feature
n_cats = 500
X_hc = pd.DataFrame({
    'cat_feature': np.random.choice([f'cat_{i}' for i in range(n_cats)], 5000),
    'numeric': np.random.randn(5000)
})
y_hc = (X_hc['cat_feature'].map(
    {f'cat_{i}': float(i % 2) for i in range(n_cats)}
)).astype(int)

print(f'High cardinality feature shape: {X_hc.shape}')
print(f'OHE would create: {n_cats - 1} columns!')
print(f'Freq encoding creates: 1 column')